# EP02 — Experimento 6: fine-tuning com QLoRA

Treina um adapter LoRA do Qwen2-VL-2B nos 1.800 exemplos de treino e avalia
contra as técnicas de test-time computing dos experimentos 1 a 5.

**Requer GPU.** `bitsandbytes` não quantiza em 4-bit na CPU, e LoRA fp32 num
modelo de 2B levaria dias. Antes de rodar: *Runtime → Change runtime type →
T4 GPU → Save*.

Todas as células são idempotentes — se a sessão cair, rode de novo na ordem e o
treino retoma do último checkpoint salvo no Drive.

## 1. Ambiente

In [ ]:
import os, shutil, subprocess

DRIVE = "/content/drive/MyDrive/vqa_project"
REPO  = "/content/ia"


def has_gpu() -> bool:
    if shutil.which("nvidia-smi") is None:
        return False
    return subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0


if not has_gpu():
    raise SystemExit(
        "Sem GPU neste runtime. Runtime > Change runtime type > T4 GPU > Save, "
        "e rode esta célula de novo."
    )
!nvidia-smi -L

In [ ]:
if not os.path.isdir(REPO):
    !git clone -q https://github.com/jojimaaa/ia.git {REPO}
else:
    !git -C {REPO} pull -q
os.chdir(f"{REPO}/00-EP02")

!pip -q install -U transformers accelerate bitsandbytes peft datasets pillow tqdm

## 2. Persistência no Drive

O adapter e os checkpoints do Trainer vão para o Drive. Sem isso, uma queda de
sessão no meio de um treino de horas perde tudo.

Só `output/checkpoints` e `output/models` são ligados ao Drive — `output` inteiro
não, porque `output/submissions/submission_constant.csv` é versionado e um
symlink em cima dele brigaria com o `git pull`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p {DRIVE}/checkpoints {DRIVE}/models {DRIVE}/submissions output
!rm -rf output/checkpoints && ln -s {DRIVE}/checkpoints output/checkpoints
!rm -rf output/models && ln -s {DRIVE}/models output/models
!ls -l output/

## 3. Sanidade antes de gastar GPU

`vqa.py` roda as asserções do parser de respostas; `exp0_constant.py` imprime o
piso de ~43% sem tocar no modelo. Se estes dois passarem, o encanamento está bom.

In [ ]:
!python data/prepare_dataset.py --no-extract
!python vqa.py
!python experiments/exp0_constant.py

## 4. Smoke test do treino

20 exemplos, para descobrir em minutos se a carga em 4-bit, o collator e a
máscara de loss funcionam. Sai num diretório separado para não sujar o
checkpoint do treino de verdade.

Observe a loss: ela tem que descer de algo em torno de 2–10 para bem menos. Loss
que não se move costuma significar máscara errada (todos os labels em -100).

In [ ]:
!python models/qwen_train.py --limit 20 --epochs 1 --save-steps 5 \
    --output-dir output/models/smoke

## 5. Treino completo

1.800 exemplos, 3 épocas, batch 1 com acumulação de 8 — ordem de 2 a 5 horas
numa T4. Se a sessão cair, refaça as células 1 a 3 e rode esta de novo: ela
detecta o último `checkpoint-*` e retoma de lá.

Para encurtar, use `--epochs 1`. Uma época já mostra se o fine-tuning bate as
técnicas de prompting, que é a pergunta do EP.

In [ ]:
!python models/qwen_train.py

## 6. Avaliação e submissão

Validação primeiro: `--val-only` mede as 200 amostras do val e para. O val é o
split correto aqui — diferente dos experimentos 1 a 5, o fine-tuning **viu** o
treino, então medir nele seria vazamento.

Lembre que o val de 200 tem IC95 de ±6,9 pontos, e que os pisos por metade são
37,9% na contagem e 52,3% na saída. Olhe as duas linhas, não só o total.

In [ ]:
!python data/generate_submission.py --val-only

In [ ]:
# Só depois de olhar os números acima: roda os 1.000 do teste e escreve o CSV.
!python data/generate_submission.py
!cp -f output/submissions/*.csv {DRIVE}/submissions/ 2>/dev/null
!ls -la {DRIVE}/submissions/

## 7. Comparação com os experimentos 1 a 5

Junta o fine-tuning e as técnicas de test-time computing na mesma tabela. As
linhas de treino e as de val não são diretamente comparáveis entre si — os
experimentos 1 a 5 foram medidos em amostras do treino (que eles nunca viram) e
o experimento 6 no val (porque viu o treino). A coluna `split` deixa isso
explícito em vez de esconder.

In [ ]:
import glob, importlib, sys
sys.path.insert(0, ".")
import vqa; importlib.reload(vqa)

ROWS = [
    ("exp1_baseline",         "output/checkpoints/exp1_baseline_train*.jsonl",         vqa.TRAIN_JSONL),
    ("exp2_prompt",           "output/checkpoints/exp2_prompt_train*.jsonl",           vqa.TRAIN_JSONL),
    ("exp3_cot",              "output/checkpoints/exp3_cot_train*.jsonl",              vqa.TRAIN_JSONL),
    ("exp4_self_consistency", "output/checkpoints/exp4_self_consistency_train*.jsonl", vqa.TRAIN_JSONL),
    ("exp5_reflexion",        "output/checkpoints/exp5_reflexion_train*.jsonl",        vqa.TRAIN_JSONL),
    ("exp6_qlora",            "output/checkpoints/exp6_lora_val.jsonl",                vqa.VAL_JSONL),
]

header = f"{'experimento':<24} {'split':>6} {'n':>5} {'total':>8} {'contagem':>10} {'saída':>8} {'s/item':>8} {'tok/item':>9}"
print(header)
print("-" * len(header))
for name, pattern, ref_path in ROWS:
    files = sorted(glob.glob(pattern))
    if not files:
        print(f"{name:<24} {'—':>6} {'—':>5}   (sem checkpoint)")
        continue
    preds = vqa.load_jsonl(files[0])
    if not preds:
        continue
    refs = {r["index"]: r for r in vqa.load_jsonl(ref_path)}
    subset = [refs[p["index"]] for p in preds if p["index"] in refs]
    m = vqa.score(preds, subset)
    tok = sum(p.get("new_tokens", 0) for p in preds) / len(preds)
    bk = m["by_kind"]
    split = "val" if ref_path == vqa.VAL_JSONL else "train"
    print(f"{name:<24} {split:>6} {m['n']:>5} {m['accuracy']:7.2f}% "
          f"{bk.get('count', {}).get('accuracy', 0):9.2f}% "
          f"{bk.get('output', {}).get('accuracy', 0):7.2f}% "
          f"{(m['avg_s'] or 0):7.2f}s {tok:8.1f}")

print()
print("pisos por constante:  contagem 37.91%   saída 52.26%   total 45.05%")